<a href="https://colab.research.google.com/github/tatahonon/cvNotebooks/blob/main/Homowork9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import spsolve


In [ ]:
def make_test_image(n, kind="smooth"):
    x = np.linspace(0.0, 1.0, n)
    y = np.linspace(0.0, 1.0, n)
    xx, yy = np.meshgrid(x, y, indexing="xy")

    if kind == "smooth":
        image = (
            0.25
            + 0.35 * xx
            + 0.20 * yy
            + 0.25 * np.exp(-80 * ((xx - 0.35) ** 2 + (yy - 0.65) ** 2))
        )
    elif kind == "waves":
        image = 0.5 + 0.25 * np.sin(4 * np.pi * xx) * np.cos(3 * np.pi * yy)
    elif kind == "rings":
        radius = np.sqrt((xx - 0.5) ** 2 + (yy - 0.5) ** 2)
        image = 0.5 + 0.35 * np.cos(18 * radius) * np.exp(-4 * radius)
    else:
        raise ValueError("Unknown image kind")

    image = image - image.min()
    image = image / image.max()
    return image.astype(np.float64)


def poisson_rhs_from_image(image):
    rhs = (
        4.0 * image[1:-1, 1:-1]
        - image[:-2, 1:-1]
        - image[2:, 1:-1]
        - image[1:-1, :-2]
        - image[1:-1, 2:]
    )
    return rhs


def residual_inf_norm(u, rhs):
    neighbours = (
        u[:-2, 1:-1]
        + u[2:, 1:-1]
        + u[1:-1, :-2]
        + u[1:-1, 2:]
    )

    residual = 4.0 * u[1:-1, 1:-1] - neighbours - rhs
    return np.max(np.abs(residual))


In [ ]:
def solve_poisson_gauss_seidel(boundary_image, rhs, tol=1e-5, max_iter=20000):
    u = np.zeros_like(boundary_image, dtype=np.float64)

    # Boundary conditions from the original image
    u[0, :] = boundary_image[0, :]
    u[-1, :] = boundary_image[-1, :]
    u[:, 0] = boundary_image[:, 0]
    u[:, -1] = boundary_image[:, -1]

    inner_shape = rhs.shape
    ii, jj = np.indices(inner_shape)

    red = (ii + jj) % 2 == 0
    black = ~red

    start = perf_counter()
    last_residual = np.inf

    for iteration in range(1, max_iter + 1):
        neighbours = (
            u[:-2, 1:-1]
            + u[2:, 1:-1]
            + u[1:-1, :-2]
            + u[1:-1, 2:]
        )

        new_values = 0.25 * (neighbours + rhs)
        inner = u[1:-1, 1:-1]
        inner[red] = new_values[red]

        neighbours = (
            u[:-2, 1:-1]
            + u[2:, 1:-1]
            + u[1:-1, :-2]
            + u[1:-1, 2:]
        )

        new_values = 0.25 * (neighbours + rhs)
        inner[black] = new_values[black]

        if iteration % 25 == 0:
            last_residual = residual_inf_norm(u, rhs)

            if last_residual < tol:
                break

    elapsed = perf_counter() - start

    return u, elapsed, iteration, last_residual


In [ ]:
def poisson_sparse_matrix(m):
    one_dimensional = diags(
        diagonals=[
            -np.ones(m - 1),
            4 * np.ones(m),
            -np.ones(m - 1)
        ],
        offsets=[-1, 0, 1],
        shape=(m, m),
        format="csr",
    )

    vertical = diags(
        diagonals=[
            -np.ones(m - 1),
            -np.ones(m - 1)
        ],
        offsets=[-1, 1],
        shape=(m, m),
        format="csr",
    )

    matrix = kron(eye(m, format="csr"), one_dimensional) + kron(
        vertical,
        eye(m, format="csr")
    )

    return matrix


def solve_poisson_sparse(boundary_image, rhs):
    m = boundary_image.shape[0] - 2
    b = rhs.copy()

    # Add boundary values to the right-hand side
    b[0, :] += boundary_image[0, 1:-1]
    b[-1, :] += boundary_image[-1, 1:-1]
    b[:, 0] += boundary_image[1:-1, 0]
    b[:, -1] += boundary_image[1:-1, -1]

    matrix = poisson_sparse_matrix(m)

    start = perf_counter()
    solution = spsolve(matrix, b.reshape(-1))
    elapsed = perf_counter() - start

    u = boundary_image.copy()
    u[1:-1, 1:-1] = solution.reshape((m, m))

    return u, elapsed


In [ ]:
sizes = [32, 48, 64, 96, 128]
kinds = ["smooth", "waves", "rings"]

results = []

for n in sizes:
    for kind in kinds:
        image = make_test_image(n, kind)
        rhs = poisson_rhs_from_image(image)

        gs_solution, gs_time, gs_iter, gs_res = solve_poisson_gauss_seidel(
            image,
            rhs
        )

        sparse_solution, sparse_time = solve_poisson_sparse(
            image,
            rhs
        )

        results.append({
            "size": n,
            "kind": kind,
            "gauss_seidel_time": gs_time,
            "sparse_time": sparse_time,
            "gauss_seidel_iterations": gs_iter,
            "gauss_seidel_residual": gs_res,
            "gauss_seidel_max_error": np.max(np.abs(gs_solution - image)),
            "sparse_max_error": np.max(np.abs(sparse_solution - image)),
        })

        print(
            f"N={n:3d}, image={kind:6s}, "
            f"GS={gs_time:.4f}s, "
            f"sparse={sparse_time:.4f}s, "
            f"GS iter={gs_iter}"
        )


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df


In [ ]:
plt.figure(figsize=(10, 6))

for method, column in [
    ("Gauss-Seidel", "gauss_seidel_time"),
    ("scipy.sparse", "sparse_time")
]:
    means = df.groupby("size")[column].mean()
    plt.plot(means.index, means.values, "o-", label=method)

plt.yscale("log")
plt.grid(True, which="both")
plt.xlabel("Image size, N x N")
plt.ylabel("Average time, seconds")
plt.title("Poisson solver time comparison")
plt.legend()
plt.show()


In [ ]:
n = 96
kind = "smooth"

image = make_test_image(n, kind)
rhs = poisson_rhs_from_image(image)

gs_solution, gs_time, gs_iter, gs_res = solve_poisson_gauss_seidel(image, rhs)
sparse_solution, sparse_time = solve_poisson_sparse(image, rhs)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gs_solution, cmap="gray", vmin=0, vmax=1)
plt.title(f"Gauss-Seidel\n{gs_time:.4f}s, {gs_iter} iter")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(sparse_solution, cmap="gray", vmin=0, vmax=1)
plt.title(f"scipy.sparse\n{sparse_time:.4f}s")
plt.axis("off")

plt.show()


Both methods reconstruct the image using the Poisson equation. The Gauss-Seidel method is simple, but it requires many iterations, so as the image size increases, it becomes noticeably slower. The scipy.sparse.linalg.spsolve method builds a sparse matrix of the system and solves it much faster, producing a very small error.